In [17]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [18]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import importlib
import unet
importlib.reload(unet)
from unet import build_unet
from unet import load_preprocess
from mobile_u_net import load_preprocess_mobilenet
from mobile_u_net import build_mobile_u_net

In [19]:
image_ds = tf.data.Dataset.list_files("data/kitti/training/image_2/*.png", shuffle=False)
mask_ds = tf.data.Dataset.list_files("data/kitti/training/semantic/*.png", shuffle=False)
dataset = tf.data.Dataset.zip((image_ds, mask_ds))

In [20]:
dataset = dataset.map(load_preprocess_mobilenet)

In [23]:
batch_size = 8
train_ds = dataset.take(160).shuffle(160).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds = dataset.skip(160).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [24]:
model = build_mobile_u_net((128,384, 3), 20)
model.summary()

/Users/kuzeyaldemir/Projects/drive-vision/mobile_u_net.py:32: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  mobile_net = tf.keras.applications.MobileNetV2(input_shape=input_shape, weights='imagenet', include_top=False)


(None, 128, 384, 20)


Model: "mobile_u_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 128, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 4, 12, 1280)    │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_4              │ (None, 8, 24, 512)     │     2,621,952 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 8, 24, 512)     │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_24 (Conv2D)              │ (None, 8, 24, 512)     │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_5              │ (None, 16, 48, 256)    │       524,544 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_25 (Conv2D)              │ (None, 16, 48, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_26 (Conv2D)              │ (None, 16, 48, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_6              │ (None, 32, 96, 128)    │       131,200 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_27 (Conv2D)              │ (None, 32, 96, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_28 (Conv2D)              │ (None, 32, 96, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_7              │ (None, 64, 192, 64)    │        32,832 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_29 (Conv2D)              │ (None, 64, 192, 64)    │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_30 (Conv2D)              │ (None, 64, 192, 64)    │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_8              │ (None, 128, 384, 64)   │        16,448 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_31 (Conv2D)              │ (None, 128, 384, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_32 (Conv2D)              │ (None, 128, 384, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_33 (Conv2D)              │ (None, 128, 384, 20)   │           660 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,882,132 (45.33 MB)

 Trainable params: 9,624,148 (36.71 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [25]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        'accuracy',
        tf.keras.metrics.MeanIoU(num_classes=20, sparse_y_pred=False, ignore_class=19, name='miou')
    ]
)

In [26]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[tf.keras.callbacks.TensorBoard(log_dir='logs/mobilenet_frozen_noskip_lr0.001')]
)

model.save('checkpoints/mobilenet_frozen_noskip_20epochs.keras')

Epoch 1/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 263ms/step - accuracy: 0.2407 - loss: 2.7718 - miou: 0.0266 - val_accuracy: 0.3461 - val_loss: 2.3334 - val_miou: 0.0246
Epoch 2/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 212ms/step - accuracy: 0.3250 - loss: 2.3373 - miou: 0.0265 - val_accuracy: 0.3236 - val_loss: 2.1779 - val_miou: 0.0185
Epoch 3/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 213ms/step - accuracy: 0.3612 - loss: 2.0676 - miou: 0.0329 - val_accuracy: 0.4940 - val_loss: 1.7856 - val_miou: 0.0555
Epoch 4/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 213ms/step - accuracy: 0.4825 - loss: 1.7159 - miou: 0.0522 - val_accuracy: 0.4950 - val_loss: 1.6443 - val_miou: 0.0532
Epoch 5/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 213ms/step - accuracy: 0.5124 - loss: 1.4623 - miou: 0.0564 - val_accuracy: 0.5117 - val_loss: 1.4898 - val_miou: 0.0553
Epoch 6/20
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 219ms/step - accuracy: 0.5168 - loss: 1.3562 - miou: 0.0576 - val_accuracy: 0.5201 - val_loss: 1.4078 - val_miou: 0.0585
Epoch 7/20
20/20 ━━━━━━━━━━

In [ ]:
for image, mask in val_ds.take(3):
    prediction = model.predict(image)
    predicted_mask = tf.argmax(prediction, axis=-1)

    plt.imshow(image[0])
    plt.show()
    plt.imshow(tf.squeeze(mask[0]), cmap='gray')
    plt.show()
    plt.imshow(predicted_mask[0], cmap='gray')
    plt.show()